# A billion tokens, and the pipeline that feeds them

Interleaved shards, a fast tokenizer, and the design decision GPT made about document boundaries.

**Runs on:** CPU — 10 minutes, mostly tokenizing · ~4 GB of disk &nbsp;·&nbsp; **Slides:** [Chapter 16 — Text Generation](../../../course-web-slides/ch16/index.html) &nbsp;·&nbsp; **Section:** 01 — Training a mini-GPT

---

## The corpus

In [ ]:
import keras
import pathlib

extract_dir = keras.utils.get_file(
    fname="mini-c4",
    origin=("https://hf.co/datasets/mattdangerw/mini-c4/resolve/main/"
            "mini-c4.zip"),
    extract=True)
extract_dir = pathlib.Path(extract_dir) / "mini-c4"

shards = sorted(extract_dir.glob("*.txt"))
print(f"{len(shards)} shards")
print(f"first shard: {shards[0].stat().st_size / 1e6:.0f} MB")

**C4** — the Colossal Clean Crawled Corpus, 750 GB in full. We take under 1%. GPT-1 used BooksCorpus, self-published books added without their authors' permission; it has since been taken down by its publishers.

## What one document looks like

In [ ]:
with open(shards[0], "r") as f:
    print(f.readline().replace("\\n", "\n")[:300])

Crawled web text: a headline, a newline, marketing copy. **Hold that image** — it is what the model's distribution will be made of, and it explains the output later in this chapter.

## SentencePiece

In [ ]:
import keras_hub
import numpy as np

vocabulary_file = keras.utils.get_file(
    origin="https://hf.co/mattdangerw/spiece/resolve/main/vocabulary.proto")
tokenizer = keras_hub.tokenizers.SentencePieceTokenizer(vocabulary_file)

print(tokenizer.tokenize("The quick brown fox."))
print(tokenizer.detokenize([450, 4996, 17354, 1701, 29916, 29889]))
print(f"\nvocabulary: {tokenizer.vocabulary_size():,}")

Exactly the byte-pair encoding from chapter 14's notebook 01, implemented in C++ and with a `detokenize()`. **Same technique, different engineering** — which is the right reason to reach for a library.

## Document boundaries as a token

In [ ]:
eot = tokenizer.token_to_id("<|endoftext|>")
print(f"<|endoftext|> is token {eot}")
print()
print("GPT makes NO attempt to keep document boundaries out of the middle")
print("of a training sample. Documents are concatenated and the boundary")
print("is marked with this token.")
print()
print("Simplicity in the pipeline, paid for with one vocabulary entry --")
print("and a recurring pattern: when a structural constraint is expensive")
print("to enforce in data, encode it as a token and let the model learn it.")

## The pipeline

In [ ]:
import tensorflow as tf

batch_size = 128
sequence_length = 256
suffix = np.array([eot])

def read_file(filename):
    ds = tf.data.TextLineDataset(filename)
    ds = ds.map(lambda x: tf.strings.regex_replace(x, r"\\n", "\n"))
    ds = ds.map(tokenizer, num_parallel_calls=8)
    return ds.map(lambda x: tf.concat([x, suffix], -1))

files = [str(f) for f in shards]
ds = tf.data.Dataset.from_tensor_slices(files)
ds = ds.interleave(read_file, cycle_length=32, num_parallel_calls=32)
ds = ds.rebatch(sequence_length + 1, drop_remainder=True)
ds = ds.map(lambda x: (x[:-1], x[1:]))
ds = ds.batch(batch_size).prefetch(8)

for x, y in ds.take(1):
    print("inputs: ", x.shape)
    print("targets:", y.shape)
    print("\noffset by one, exactly as in chapter 15:")
    print(" x[0][:8] =", x[0][:8].numpy())
    print(" y[0][:8] =", y[0][:8].numpy())
    break

**`interleave`** lets every CPU core tokenize a different shard simultaneously. **`rebatch(257)`** windows the token stream into even samples. **`prefetch(8)`** keeps batches ready so the GPU never waits — chapter 18 returns to this as the thing that turns eight expensive GPUs into eight expensive idle GPUs.

## The size of it

In [ ]:
num_batches = 29373        # counted once; see the note below
num_val_batches = 500
num_train_batches = num_batches - num_val_batches

val_ds = ds.take(num_val_batches).repeat()
train_ds = ds.skip(num_val_batches).repeat()

print(f"{num_batches:,} batches x {batch_size} samples x "
      f"{sequence_length} tokens")
print(f"= {num_batches * batch_size * sequence_length / 1e9:.2f} billion tokens")
print()
print("Counting it yourself:  ds.reduce(0, lambda c, _: c + 1)")
print("...but tokenizing a dataset this size takes several minutes on a")
print("fast CPU, so the number is hardcoded above.")

## Sanity-check the pipeline before spending six hours on it

In [ ]:
for x, y in train_ds.take(1):
    sample = x[0].numpy()
    print("decoded first sample:\n")
    text = tokenizer.detokenize(sample)
    text = text.numpy().decode() if hasattr(text, "numpy") else str(text)
    print(text[:600])
    print("\n---")
    print("does it contain a document boundary?",
          bool((sample == eot).any()))
    break

**Read the decoded text before training.** A pipeline bug here — a wrong regex, a misaligned offset — costs six hours of GPU time and produces a model that trains happily on nonsense.

---

## What to take away

- `interleave` parallelises tokenization across shards; `prefetch` keeps the accelerator fed.
- GPT concatenates documents and marks boundaries with a token rather than respecting them in the data.
- The offset-by-one split is chapter 15's, unchanged.
- Decode a sample and read it before starting an expensive run.